In [6]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import handle_missing_values, remove_duplicates, remove_outliers_iqr, drop_unnecessary, map_owner

df = pd.read_csv(PROJECT_ROOT / "data/raw/car-prices.csv")
print(df.columns)
df.head()
df = drop_unnecessary(df, ['name'])
df = map_owner(df)
df = handle_missing_values(df)
df = remove_duplicates(df)
df = remove_outliers_iqr(df, ['selling_price', 'km_driven', 'year'])

df.shape

Index(['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner'],
      dtype='str')


(3268, 7)

In [7]:
import src.data_cleaning as c 
print(dir(c))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'drop_unnecessary', 'handle_missing_values', 'load_data', 'map_owner', 'np', 'pd', 'remove_duplicates', 'remove_outliers_iqr']


In [8]:
from src.preprocessing import encode_categorical, split_data, scale_features
df_encoded = encode_categorical(df, ['fuel', 'seller_type', 'transmission'])
X_train, X_test, y_train, y_test = split_data(df_encoded)
numeric_cols = ['year', 'km_driven']
X_train, X_test, scaler = scale_features(X_train, X_test, numeric_cols)

X_train.shape, X_test.shape

((2614, 10), (654, 10))

In [9]:
df.to_csv('../data/processed/cars-clean.csv')
print("Cleaned and saved sucessfully")

Cleaned and saved sucessfully


In [10]:
import numpy
import scipy
import sklearn
print(numpy.__version__)
print(scipy.__version__)
print(sklearn.__version__)

2.5.3
1.18.1
1.9.1


In [11]:
from pathlib import Path
import sys
import pandas as pd
from src.preprocessing import encode_categorical, split_data
from src.train_models import build_pipeline, evaluate_model, train_all_models

df = pd.read_csv("../data/processed/cars-clean.csv")
df.head()

,Unnamed: 0,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,0,2007.0,60000,70000.0,Petrol,Individual,Manual,1.0
1,1,2007.0,135000,50000.0,Petrol,Individual,Manual,1.0
2,2,2012.0,600000,100000.0,Diesel,Individual,Manual,1.0
3,3,2017.0,250000,46000.0,Petrol,Individual,Manual,1.0
4,4,2014.0,450000,141000.0,Diesel,Individual,Manual,2.0


In [12]:
df_encoded = encode_categorical(df, ['fuel', 'seller_type', 'transmission'])
df_encoded.head()

,Unnamed: 0,year,selling_price,km_driven,owner,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual
0,0,2007.0,60000,70000.0,1.0,False,False,False,True,True,False,True
1,1,2007.0,135000,50000.0,1.0,False,False,False,True,True,False,True
2,2,2012.0,600000,100000.0,1.0,True,False,False,False,True,False,True
3,3,2017.0,250000,46000.0,1.0,False,False,False,True,True,False,True
4,4,2014.0,450000,141000.0,2.0,True,False,False,False,True,False,True


In [13]:
X_train, X_test, y_train, y_test = split_data(df_encoded)
numeric_cols = ['year', 'km_driven']
X_train.shape, X_test.shape

((2614, 11), (654, 11))

In [14]:
print(X_train.dtypes)

Unnamed: 0                        int64
year                            float64
km_driven                       float64
owner                           float64
fuel_Diesel                        bool
fuel_Electric                      bool
fuel_LPG                           bool
fuel_Petrol                        bool
seller_type_Individual             bool
seller_type_Trustmark Dealer       bool
transmission_Manual                bool
dtype: object


In [16]:
results_df, trained_models = train_all_models(
    X_train, y_train, X_test, y_test, numeric_cols
)
results_df

Entrainement: LinearRegression
scores: {'RMSE': np.float64(179041.09460749355), 'MAE': 134249.97851942253, 'R2': 0.527677610454341}


,RMSE,MAE,R2
LinearRegression,179041.094607,134249.978519,0.527678
